# Introdução 

## Contexto

O serviço de telefonia virtual CallMeMaybe está desenvolvendo uma nova funcionalidade que permitirá aos supervisores **identificar operadores menos eficientes**.

Os clientes da empresa são **organizações que precisam gerenciar grandes volumes de chamadas** — tanto recebidas quanto realizadas por diversos operadores.

De acordo com as regras de negócio, um operador é considerado **ineficiente** se:

- Possui **muitas chamadas recebidas perdidas** (internas ou externas);
- Apresenta **tempo de espera prolongado** nas chamadas recebidas;
- E, no caso de operadores responsáveis por chamadas de saída, realiza **poucas chamadas ativas**.

## Dados

O dataset compactado **`telecom_dataset_us.csv`** contém as seguintes colunas:

- **`user_id`**: ID da conta do cliente
- **`date`**: data em que as estatísticas foram coletadas
- **`direction`**: “direção” da chamada (`out` para chamadas **saídas**, `in` para chamadas **entrantes**)
- **`internal`**: indica se a chamada foi **interna** (entre operadores de um mesmo cliente)
- **`operator_id`**: identificador do operador
- **`is_missed_call`**: indica se foi uma **chamada perdida**
- **`calls_count`**: número de chamadas
- **`call_duration`**: duração da chamada (sem incluir o tempo de espera)
- **`total_call_duration`**: duração total da chamada (incluindo o tempo de espera)

O conjunto de dados **`telecom_clients_us.csv`** contém as seguintes colunas:

- **`user_id`**: ID do cliente
- **`tariff_plan`**: plano tarifário atual do cliente
- **`date_start`**: data de registro do cliente

## Objetivo 

O objetivo da análise será averiguar a eficiência dos operadores de acordo com as orientações base fornecidas pela companhia para oferecer ao fim uma avaliação confiável para tomada de decisão.

# Setup e Dados

## Ambiente

### Importação bibliotecas

In [1]:
# importação de bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import stats as st 
import warnings 

### Configurações Globais

In [2]:
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10,6)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

### Carregamento de dados

In [3]:
# leitura primeiro dataset
telecom = pd.read_csv('../datasets/telecom_dataset_new.csv')

In [4]:
# leitura segundo dataset
clients = pd.read_csv('../datasets/telecom_clients.csv')

## Pré Processamento 

### Primeiras impressões

In [19]:
# primeira visualização
telecom.head()

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.00,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.00,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.00,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.00,True,3,0,25


In [20]:
telecom.info()

<class 'pandas.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  str    
 2   direction            53902 non-null  str    
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(1), str(2)
memory usage: 3.3+ MB


In [21]:
# descoberta informações
telecom.describe().T

,count,mean,std,min,25%,50%,75%,max
user_id,53902.00,167295.34,598.88,166377.00,166782.00,167162.00,167819.00,168606.00
operator_id,45730.00,916535.99,21254.12,879896.00,900788.00,913938.00,937708.00,973286.00
calls_count,53902.00,16.45,62.92,1.00,1.00,4.00,12.00,4817.00
call_duration,53902.00,866.68,3731.79,0.00,0.00,38.00,572.00,144395.00
total_call_duration,53902.00,1157.13,4403.47,0.00,47.00,210.00,902.00,166155.00


O dataset *telecom* possui a nomenclatura de suas colunas dentro do padrão snake_case, 

In [22]:
# descoberta informações
clients.head()

,user_id,tariff_plan,date_start
0,166713,A,2019-08-15
1,166901,A,2019-08-23
2,168527,A,2019-10-29
3,167097,A,2019-09-01
4,168193,A,2019-10-16


In [23]:
# descoberta características dos dados  
clients.info()

<class 'pandas.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      732 non-null    int64
 1   tariff_plan  732 non-null    str  
 2   date_start   732 non-null    str  
dtypes: int64(1), str(2)
memory usage: 17.3 KB


In [24]:
# descoberta características dos dados  
clients.describe().T

,count,mean,std,min,25%,50%,75%,max
user_id,732.00,167431.93,633.81,166373.00,166900.75,167432.00,167973.00,168606.00


# Análise Exploratória de Dados

In [25]:
telecom['date'].min()

'2019-08-02 00:00:00+03:00'

In [26]:
telecom['date'].max()

'2019-11-28 00:00:00+03:00'

In [ ]:
telecom['user_id'].nunique()


307

In [28]:
clients['user_id'].nunique()

732

# Cálculo KPIs e Identificação de Ineficiência 

# Conclusão e Recomendações de Negócio